# MIMIC-III EDA: Per-Patient Statistical Analysis & Visualization

Parts of this Notebook:
1. **Data loading** – INPUTEVENTS_MV, ICUSTAYS, D_ITEMS
2. **Visualization** – replicates the reference scatter plot (LOESS trend, continuous blue colour scale for item IDs, ggplot2-style theme)
3. **Statistical analysis** – descriptive stats aggregated hierarchically: ICU stay → hospital admission → patient
4. **Export** – saves plots as PNG and stats as CSV

> **Required files** (`DATA_DIR`):
> `INPUTEVENTS_MV.csv`, `ICUSTAYS.csv`, `D_ITEMS.csv`

In [ ]:
import subprocess, sys

import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from statsmodels.nonparametric.smoothers_lowess import lowess

warnings.filterwarnings("ignore")

In [ ]:
DATA_DIR   = ".\data\datasets" 
OUTPUT_DIR = ".\output" 

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1 - Load Data

In [ ]:
print("Loading INPUTEVENTS_MV.csv")
inputs = pd.read_csv(
    os.path.join(DATA_DIR, "INPUTEVENTS_MV.csv"),
    usecols=["SUBJECT_ID", "HADM_ID", "ICUSTAY_ID",
             "STARTTIME", "ITEMID", "AMOUNT", "AMOUNTUOM"],
    parse_dates=["STARTTIME"],
    low_memory=False,
)

print("Loading ICUSTAYS.csv")
icustays = pd.read_csv(
    os.path.join(DATA_DIR, "ICUSTAYS.csv"),
    usecols=["SUBJECT_ID", "HADM_ID", "ICUSTAY_ID", "INTIME", "OUTTIME"],
    parse_dates=["INTIME", "OUTTIME"],
    low_memory=False,
)

print("Loading D_ITEMS.csv")
d_items = pd.read_csv(
    os.path.join(DATA_DIR, "D_ITEMS.csv"),
    usecols=["ITEMID", "LABEL", "CATEGORY"],
    low_memory=False,
)

print(f"\nINPUTEVENTS_MV : {len(inputs):>10,} rows")
print(f"ICUSTAYS        : {len(icustays):>10,} rows")
print(f"D_ITEMS         : {len(d_items):>10,} rows")

## 3 - The graph

In [ ]:
def plot_icu_stay(
    icustay_id: int,
    inputs_df: pd.DataFrame,
    icustays_df: pd.DataFrame,
    d_items_df: pd.DataFrame,
    day: int = 1,
    save: bool = True,
):
    """
    Parameters
    ----------
    icustay_id  : ICUSTAY_ID to plot
    inputs_df   : INPUTEVENTS_MV DataFrame
    icustays_df : ICUSTAYS DataFrame
    d_items_df  : D_ITEMS DataFrame
    day         : which 24-h window to show (1 = first day after admission)
    save        : if True, save PNG to OUTPUT_DIR
    """
    # locate the patients stay 
    stay_row = icustays_df[icustays_df["ICUSTAY_ID"] == icustay_id]
    if stay_row.empty:
        print(f"ICUSTAY_ID {icustay_id} not found.")
        return

    intime    = stay_row["INTIME"].iloc[0]
    day_start = intime + pd.Timedelta(days=day - 1)

    # filter events for the time window
    ev = inputs_df[inputs_df["ICUSTAY_ID"] == icustay_id].copy()
    ev = ev.merge(d_items_df[["ITEMID", "LABEL"]], on="ITEMID", how="left")
    ev["Relative_Day"] = (
        (ev["STARTTIME"] - day_start).dt.total_seconds() / 86_400
    )
    ev = ev[(ev["Relative_Day"] >= 0) & (ev["Relative_Day"] <= 1.0)]

    if ev.empty:
        print(f"No events for ICUSTAY_ID {icustay_id} on day {day}.")
        return

    id_min, id_max = ev["ITEMID"].min(), ev["ITEMID"].max()
    norm = mcolors.Normalize(vmin=id_min, vmax=id_max)

    # LOESS + confidence band
    xy = ev[["Relative_Day", "AMOUNT"]].dropna()
    xy = xy[xy["AMOUNT"] > 0].sort_values("Relative_Day")  # drops zeros/negatives
    
    # Calculates main LOESS (it=0 forces it to match R's default behavior)
    smooth = lowess(xy["AMOUNT"], xy["Relative_Day"], frac=0.35, it=0, return_sorted=True)
    xs, ys = smooth[:, 0], smooth[:, 1] 

    # Bootstrap for confidence bands 
    rng = np.random.default_rng(42)
    boot_ys = []
    for _ in range(200):
        idx = rng.integers(0, len(xy), size=len(xy))
        bxy = xy.iloc[idx].sort_values("Relative_Day")
        bs  = lowess(bxy["AMOUNT"], bxy["Relative_Day"], frac=0.75, it=0, return_sorted=True)
        boot_ys.append(np.interp(xs, bs[:, 0], bs[:, 1])) 

    band  = np.array(boot_ys)
    lower = np.percentile(band, 5,  axis=0)
    upper = np.percentile(band, 95, axis=0)

    # figure
    fig, ax = plt.subplots(figsize=(9, 6), facecolor="white")
    ax.set_facecolor("#EBEBEB")
    ax.grid(color="white", linewidth=0.8)
    ax.set_axisbelow(True)
   
    sc = ax.scatter(
        ev["Relative_Day"], ev["AMOUNT"],
        c=ev["ITEMID"], cmap="Blues", norm=norm,
        s=55, edgecolors="none", zorder=3, alpha=0.9,
    )

    # confidence band & LOESS line
    if xs is not None:
        ax.fill_between(xs, lower, upper, color="grey", alpha=0.35, zorder=2)
        ax.plot(xs, ys, color="royalblue", linewidth=2.0, zorder=4)
    # colorbar
    cbar = fig.colorbar(sc, ax=ax, pad=0.02, shrink=0.85)
    cbar.set_label("It", fontsize=11)
    cbar.set_ticks(np.linspace(id_min, id_max, 5, dtype=int))
    cbar.ax.tick_params(labelsize=8)

    ax.set_title(f"ICU_{icustay_id}", fontsize=13,
                 fontweight="bold", loc="left")
    ax.set_xlabel("Time",   fontsize=11)
    ax.set_ylabel("Values", fontsize=11)
    ax.set_xlim(0, 1)
    ax.xaxis.set_major_locator(plt.MultipleLocator(0.25))

    plt.tight_layout()

    if save:
        path = os.path.join(OUTPUT_DIR, f"ICU_{icustay_id}_day{day}.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        print(f"Plot saved → {path}")

    plt.show()
    plt.close()

### 3.1 · Example – ICUSTAY_ID 200072 (Day 1)

In [ ]:
plot_icu_stay(200072, inputs, icustays, d_items, day=1, save=True)

## 4 · Statistical Analysis

Descriptive statistics are computed at **three hierarchical levels**:

```
SUBJECT_ID
  └── HADM_ID  (hospital admission)
        └── ICUSTAY_ID  (ICU stay)
```

Each level aggregates: total amount, mean/median/std dose, event count, and (at the ICU level) amount per LOS hour.

In [ ]:
def compute_statistics(inputs_df, d_items_df, icustays_df):
    """
    Returns
    -------
    icu_stats       : pd.DataFrame  - per ICUSTAY_ID x ITEMID
    admission_stats : pd.DataFrame  - per HADM_ID    x ITEMID
    patient_stats   : pd.DataFrame  - per SUBJECT_ID x ITEMID
    """
    df = (
        inputs_df
        .merge(d_items_df[["ITEMID", "LABEL", "CATEGORY"]],
               on="ITEMID", how="left")
        .merge(icustays_df[["ICUSTAY_ID", "INTIME", "OUTTIME"]],
               on="ICUSTAY_ID", how="left")
    )
    df["LOS_hours"] = (
        (df["OUTTIME"] - df["INTIME"]).dt.total_seconds() / 3600
    )

    # ICU stay × item
    icu_stats = (
        df.groupby(["SUBJECT_ID", "HADM_ID", "ICUSTAY_ID", "ITEMID", "LABEL"])
        .agg(
            Total_Amount  = ("AMOUNT", "sum"),
            Mean_Dose     = ("AMOUNT", "mean"),
            Median_Dose   = ("AMOUNT", "median"),
            Std_Dose      = ("AMOUNT", "std"),
            Min_Dose      = ("AMOUNT", "min"),
            Max_Dose      = ("AMOUNT", "max"),
            Event_Count   = ("AMOUNT", "count"),
            LOS_hours     = ("LOS_hours", "first"),
        )
        .reset_index()
    )
    icu_stats["Amount_per_LOS_hour"] = (
        icu_stats["Total_Amount"]
        / icu_stats["LOS_hours"].replace(0, np.nan)
    )

    # Hospital admission × item
    admission_stats = (
        icu_stats
        .groupby(["SUBJECT_ID", "HADM_ID", "ITEMID", "LABEL"])
        .agg(
            Total_Amount_Admission  = ("Total_Amount",  "sum"),
            Mean_Per_Stay           = ("Total_Amount",  "mean"),
            Std_Per_Stay            = ("Total_Amount",  "std"),
            Total_Events_Admission  = ("Event_Count",   "sum"),
            N_ICU_Stays             = ("ICUSTAY_ID",    "nunique"),
        )
        .reset_index()
    )

    # Patient × item
    patient_stats = (
        admission_stats
        .groupby(["SUBJECT_ID", "ITEMID", "LABEL"])
        .agg(
            Total_Amount_All        = ("Total_Amount_Admission", "sum"),
            Mean_Per_Admission      = ("Total_Amount_Admission", "mean"),
            Std_Per_Admission       = ("Total_Amount_Admission", "std"),
            Total_Events_All        = ("Total_Events_Admission", "sum"),
            N_Admissions            = ("HADM_ID",                "nunique"),
            N_ICU_Stays_All         = ("N_ICU_Stays",            "sum"),
        )
        .reset_index()
    )

    return icu_stats, admission_stats, patient_stats

In [ ]:
icu_stats, admission_stats, patient_stats = compute_statistics(
    inputs, d_items, icustays
)
print(f"ICU-level stats      : {len(icu_stats):,} rows")
print(f"Admission-level stats: {len(admission_stats):,} rows")
print(f"Patient-level stats  : {len(patient_stats):,} rows")

### 4.1 - Specific Patient Top-5 Items Summary

In [ ]:
TARGET_SUBJECT_ID = 23 
TOP_N = 5 

# Filter the dataframe for just the selected patient
patient_df = patient_stats[patient_stats["SUBJECT_ID"] == TARGET_SUBJECT_ID]

if not patient_df.empty:
    # Get the top N rows for this specific patient
    top = patient_df.nlargest(TOP_N, "Total_Amount_All")
    
    # Extract the admission and ICU counts
    n_adm  = patient_df["N_Admissions"].iloc[0]
    n_icu  = patient_df["N_ICU_Stays_All"].iloc[0]
    
    print(f"\nSUBJECT_ID {TARGET_SUBJECT_ID}  -  {n_adm} admission(s), {n_icu} ICU stay(s)")
    
    # Display the formatted table
    display(
        top[["LABEL", "Total_Amount_All",
             "Mean_Per_Admission", "Std_Per_Admission",
             "Total_Events_All"]]
        .rename(columns={
            "Total_Amount_All":   "Total Amount",
            "Mean_Per_Admission": "Mean / Admission",
            "Std_Per_Admission":  "Std / Admission",
            "Total_Events_All":   "# Events",
        })
        .reset_index(drop=True)
    )
else:
    print(f"SUBJECT_ID {TARGET_SUBJECT_ID} not found in the dataset.")

## 5 - Export Results

In [ ]:
icu_stats.to_csv(
    os.path.join(OUTPUT_DIR, "stats_icu_level.csv"), index=False)
admission_stats.to_csv(
    os.path.join(OUTPUT_DIR, "stats_admission_level.csv"), index=False)
patient_stats.to_csv(
    os.path.join(OUTPUT_DIR, "stats_patient_level.csv"), index=False)

print("Exported:")
print(f"  stats_icu_level.csv       ({len(icu_stats):,} rows)")
print(f"  stats_admission_level.csv ({len(admission_stats):,} rows)")
print(f"  stats_patient_level.csv   ({len(patient_stats):,} rows)")